# Embedding

## Reading and Preprocessing data

In [153]:
MED_DATA_PATH = "./TP_ISD2020/QUAERO_FrenchMed/QUAERO_FrenchMed_traindev.ospl"
PRESS_DATA_PATH = "./TP_ISD2020/QUAERO_FrenchPress/QUAERO_FrenchPress_traindev.ospl"

In [154]:
# Load the data file 
def load_data(Path):
    with open(Path) as f:
        data = f.readlines()
    return data

# Preprocess the text by tokenizing it into the input format of the models
def preprocess_data(data):
    tokenized_data = [line.lower().strip().split() for line in data]
    return tokenized_data

In [155]:
# Load and preprocess the data

med_data = load_data(MED_DATA_PATH)
press_data = load_data(PRESS_DATA_PATH)

tokenized_med_data = preprocess_data(med_data)
tokenized_press_data = preprocess_data(press_data)

In [156]:
# Preview of the results for medical data
tokenized_med_data[0:5]  # Display the first 5 tokenized sentences from the medical data

[['emea', '/', 'h', '/', 'c', '/', '551'],
 ['prialt'],
 ['qu', '’', 'est', 'ce', 'que', 'prialt', '?'],
 ['prialt',
  'est',
  'une',
  'solution',
  'pour',
  'perfusion',
  'contenant',
  'le',
  'principe',
  'actif',
  'ziconotide',
  ',',
  'à',
  'des',
  'concentrations',
  'de',
  '100',
  'ou',
  '25',
  'microgrammes',
  'par',
  'millilitre',
  '.'],
 ['dans', 'quel', 'cas', 'prialt', 'est', '-', 'il', 'utilisé', '?']]

In [157]:
# Preview of the results for press data
tokenized_press_data[0:5]  # Display the first 5 tokenized sentences from the medical data

[['patricia',
  'martin',
  ',',
  'que',
  'voici',
  ',',
  'que',
  'voilà',
  '!',
  'oh',
  ',',
  'bonjour',
  'nicolas',
  'stoufflet',
  '.'],
 ['france', 'inter', ',', '7', 'heures', '.'],
 ['le', 'journal', ',', 'simon', 'tivolle', '.'],
 ['bonjour', '!', 'lundi', '7', 'décembre', '.'],
 ['deux',
  'incendies',
  'cette',
  'nuit',
  'en',
  'région',
  'parisienne',
  ',',
  'dans',
  'une',
  'maison',
  'de',
  'retraite',
  'de',
  'livry-gargan',
  'en',
  'seine-saint-denis',
  ',',
  '7',
  'personnes',
  'ont',
  'péri',
  'dans',
  'les',
  'flammes',
  '.']]

## Word Embeddings Training 

### Word2Vec Models

In [174]:
from gensim.models import Word2Vec

# Helper function to train Word2Vec models with specified parameters
def w2v(sentences, sg, vector_size=100, window=5):

	model = Word2Vec(
		sentences=sentences,
		vector_size=vector_size,
		window=window,
		min_count=1,
		sg=sg,
		seed=42,
		workers=1,
  		epochs=30
	)
	return model

# CBOW model for medical data
w2v_cbow_med_model = w2v(tokenized_med_data, sg=0)

# Skipgram model for medical data
w2v_skipgram_med_model = w2v(tokenized_med_data, sg=1)

# CBOW model for press data
w2v_cbow_press_model = w2v(tokenized_press_data, sg=0)

# Skipgram model for press data
w2v_skipgram_press_model = w2v(tokenized_press_data, sg=1)


### Fast Text models

In [175]:
from gensim.models import FastText

# Helper function to create a FastText model with the specified parameters
def fasttext_cbow(sentences, vector_size=100, window=5, sg=0):
    model = FastText(
		sentences=sentences,
		vector_size=vector_size,
		window=window,
		min_count=1,
		sg=sg,
		seed=42,
		workers=1,
		epochs=30
	)
    return model

# CBOW model for medical data using FastText
ft_cbow_med_model = fasttext_cbow(tokenized_med_data)

# CBOW model for press data using FastText
ft_cbow_press_model = fasttext_cbow(tokenized_press_data)

### Saving the embeddings

In [176]:
import os

os.makedirs("embeddings", exist_ok=True)

all_models = {
    "w2v_cbow_med": w2v_cbow_med_model,
    "w2v_skipgram_med": w2v_skipgram_med_model,
    "ft_cbow_med": ft_cbow_med_model,
    "w2v_cbow_press": w2v_cbow_press_model,
    "w2v_skipgram_press": w2v_skipgram_press_model,
    "ft_cbow_press": ft_cbow_press_model,
}

for name, model in all_models.items():
    model.wv.save_word2vec_format(f"embeddings/{name}.bin", binary=True)

### Sanity check

In [177]:
# Sanity check function to verify the models and their word vectors

med_models = [w2v_cbow_med_model, w2v_skipgram_med_model, ft_cbow_med_model]
press_models = [w2v_cbow_press_model, w2v_skipgram_press_model, ft_cbow_press_model]

def sanityquick_check(models, words):
	for model in models:
		print(f"Model: {model}")
		for word in words:
			if word in model.wv:
				print(f"Word: {word}, Vector: {model.wv[word][:5]}, Vector Length: {len(model.wv[word])}")
			else:
				print(f"Word: {word} not found in the model.")
		print("\n")
  

In [178]:
# Sanity check for medical models with a list of relevant words
sanityquick_check(med_models, ["cancer", "santé", "maladie", "traitement", "virus"])

Model: Word2Vec<vocab=8088, vector_size=100, alpha=0.025><vocab=8088, vector_size=100, alpha=0.025><vocab=8088, vector_size=100, alpha=0.025>
Word: cancer, Vector: [-0.1765881   0.04514957  0.06061367  0.379157   -0.03219745], Vector Length: 100
Word: santé, Vector: [ 0.04032586 -0.13171193 -0.304106    0.17654017  0.0185817 ], Vector Length: 100
Word: maladie, Vector: [-0.7445137   0.26759884  0.22162245  0.7627148   0.08127327], Vector Length: 100
Word: traitement, Vector: [ 0.08166762  0.8961904  -0.7866601   0.8993067  -0.232611  ], Vector Length: 100
Word: virus, Vector: [ 0.15060808 -0.292084   -0.00368749  0.3839941  -0.07450192], Vector Length: 100


Model: Word2Vec<vocab=8088, vector_size=100, alpha=0.025><vocab=8088, vector_size=100, alpha=0.025><vocab=8088, vector_size=100, alpha=0.025>
Word: cancer, Vector: [ 0.08408087  0.25958043  0.36585295  0.37368315 -0.0771578 ], Vector Length: 100
Word: santé, Vector: [-0.03400959 -0.06919105 -0.22947569  0.30954584  0.02922747], Vec

In [179]:
# Sanity check for press models with a list of relevant words
sanityquick_check(press_models, ["politique", "économie", "sport", "culture", "technologie"])

Model: Word2Vec<vocab=8088, vector_size=100, alpha=0.025><vocab=8088, vector_size=100, alpha=0.025><vocab=38279, vector_size=100, alpha=0.025>
Word: politique, Vector: [ 1.4350159  -6.5615673  -0.27770916  1.4482448   3.2408607 ], Vector Length: 100
Word: économie, Vector: [ 0.2226019  -1.7169805   0.27259317  3.0243497  -0.9289707 ], Vector Length: 100
Word: sport, Vector: [-1.6467375  -0.16675922 -0.6862458   1.0654593  -1.8279684 ], Vector Length: 100
Word: culture, Vector: [ 0.25639504 -2.9768286   0.3620635   0.69463164  0.1902515 ], Vector Length: 100
Word: technologie, Vector: [ 0.5955603  -0.89298344 -0.13178909 -0.04220592  0.53308684], Vector Length: 100


Model: Word2Vec<vocab=8088, vector_size=100, alpha=0.025><vocab=8088, vector_size=100, alpha=0.025><vocab=38279, vector_size=100, alpha=0.025>
Word: politique, Vector: [-0.12765251  0.10559668 -0.18848711  0.36784756  0.29849917], Vector Length: 100
Word: économie, Vector: [ 0.31589445 -0.15555838  0.38792408  0.54348975 -0

## Semantic Similarity

### Comparaison of the embeddings trained on the same corpus

In [180]:
cadidat_words = ['patient', 'traitement', 'maladie', 'solution', 'jaune']

# Function to get word embeddings for a list of words from multiple models

def get_word_embeddings(models, words):
	embeddings = {}
	for model in models:
		embeddings[model] = {}
		for word in words:
			if word in model.wv:
				embeddings[model][word] = model.wv[word]
			else:
				embeddings[model][word] = None
	return embeddings



In [181]:
# Get word embeddings for the candidate words from the medical models
word_embeddings_med = get_word_embeddings(med_models, cadidat_words)

In [182]:
def compare_embeddings(embeddings):
	for model, words in embeddings.items():
		print(f"Model: {model}")
		word_embeddings = [embeddings[model][word] for word in cadidat_words]

In [183]:
# 10 Most similar words for each candidate word in the medical models

def get_most_similar_words(models, words, topn=10):
	model_results = []
 
	for model in models:
		similar_words = {}
		for word in words:
			if word in model.wv:
				similar_words[word] = model.wv.most_similar(word, topn=topn)
			else:
				similar_words[word] = None
		model_results.append(similar_words)
	return model_results

In [184]:
import pandas as pd
# CBOW model results for the candidate words in the medical models
med_model_results = get_most_similar_words(med_models, cadidat_words)

cbow_similar_words = pd.DataFrame(med_model_results[0])
cbow_similar_words

,patient,traitement,maladie,solution,jaune
0,"(tysabri, 0.8929116129875183)","(instauration, 0.8599756956100464)","(parkinson, 0.9248817563056946)","(poudre, 0.942267656326294)","(oxyde, 0.9779549241065979)"
1,"(qui, 0.891009509563446)","(début, 0.8460553288459778)","(coronarienne, 0.8962394595146179)","(injectable, 0.9293431043624878)","(serti, 0.9762580394744873)"
2,"(avoir, 0.8877918720245361)","(risque, 0.8345841765403748)","(antirétroviraux, 0.8778077363967896)","(dissoudre, 0.9251070022583008)","(caoutchouc, 0.9759230613708496)"
3,"(cathéter, 0.8816549181938171)","(diagnostic, 0.8042226433753967)","(infection, 0.8738994598388672)","(buvable, 0.9212127923965454)","(réunion, 0.9758464694023132)"
4,"(délai, 0.8814948201179504)","(définitif, 0.8027284145355225)","(marfan, 0.8723532557487488)","(contient, 0.9144995212554932)","(ix, 0.9718866348266602)"
5,"(répondu, 0.8796426653862)","(arrêt, 0.7924464344978333)","(juger, 0.8684974312782288)","(perfusable, 0.9143097996711731)","(financées, 0.9709644913673401)"
6,"(envisagée, 0.8762686252593994)","(débuter, 0.788699746131897)","(leptospirose, 0.8643409609794617)","(lépirudine, 0.9123354554176331)","(trou, 0.9705710411071777)"
7,"(prescrire, 0.8756699562072754)","(contrôle, 0.7864606380462646)","(charge, 0.8619171977043152)","(conditionnée, 0.9119361639022827)","(méthyle, 0.9693435430526733)"
8,"(allaiter, 0.8744040727615356)","(opportuniste, 0.7832421064376831)","(décision, 0.8574418425559998)","(300, 0.9114261865615845)","(1925, 0.9686945676803589)"
9,"(interrompre, 0.8730151653289795)","(intrarachidien, 0.7792053818702698)","(tension, 0.8557232022285461)","(intraveineuse, 0.9063240885734558)","(macrogol, 0.9682480096817017)"


In [185]:
# Skipgram model results for the candidate words in the medical models
skipgram_similar_words = pd.DataFrame(med_model_results[1])
skipgram_similar_words

,patient,traitement,maladie,solution,jaune
0,"(carte, 0.831990659236908)","(traités, 0.5707740187644958)","(liée, 0.8042786717414856)","(limpide, 0.8798345923423767)","(pâle, 0.8650153279304504)"
1,"(montrez, 0.8038404583930969)","(concomitant, 0.5509161353111267)","(parkinson, 0.7863666415214539)","(incolore, 0.8719403147697449)","(oxyde, 0.8370767831802368)"
2,"(spéciale, 0.7946785688400269)","(opportuniste, 0.539807915687561)","(recklinghausen, 0.7859011292457581)","(diluer, 0.8690229058265686)","(talc, 0.823702871799469)"
3,"(alerte, 0.7812373042106628)","(prostatique, 0.5371869802474976)","(crohn, 0.7726890444755554)","(buvable, 0.8604428768157959)","(fer, 0.8190636038780212)"
4,"(montrer, 0.7323099374771118)","(antérieur, 0.5371071696281433)","(bourneville, 0.7598428726196289)","(réchauffer, 0.8586578369140625)","(méthylhydroxypropylcellulose, 0.8153849244117..."
5,"(avoir, 0.7291834354400635)","(chirurgical, 0.5367563366889954)","(avancé, 0.7525296807289124)","(lepirudine, 0.8584926724433899)","(orange, 0.806239128112793)"
6,"(entraîné, 0.7212455868721008)","(surveillée, 0.5349908471107483)","(thrombo, 0.7267509698867798)","(poudre, 0.8430634140968323)","(triacétine, 0.8032900094985962)"
7,"(répondu, 0.709577202796936)","(instauration, 0.5345603823661804)","(thromboembolique, 0.7257395386695862)","(ambiante, 0.8425248861312866)","(laurylsulfate, 0.7955158948898315)"
8,"(étroitement, 0.709269642829895)","(débuter, 0.5334835052490234)","(développant, 0.7234809994697571)","(dissoudre, 0.8418287038803101)","(polyéthylène, 0.7948690056800842)"
9,"(aptitude, 0.7059157490730286)","(suivi, 0.5278658270835876)","(avancée, 0.7206597924232483)","(injectable, 0.8350393772125244)","(blanc, 0.7939554452896118)"


In [186]:
# FastText model results for the candidate words in the medical models
fasttext_similar_words = pd.DataFrame(med_model_results[2])
fasttext_similar_words

,patient,traitement,maladie,solution,jaune
0,"(patiente, 0.9720413684844971)","(taaitement, 0.9884245991706848)","(malade, 0.9744800925254822)","(dissolution, 0.989467442035675)","(burhenne, 0.9683135747909546)"
1,"(aient, 0.954264760017395)","(traitment, 0.9815952181816101)","(malaise, 0.9249409437179565)","(evolution, 0.9617990851402283)","(banane, 0.9672237038612366)"
2,"(étaient, 0.9520055055618286)","(évitement, 0.9812362194061279)","(partenaire, 0.9232379198074341)","(dilution, 0.9617385268211365)","(linnéenne, 0.9607348442077637)"
3,"(parvient, 0.9513995051383972)","(allaitement, 0.9776328802108765)","(mémoire, 0.9226042628288269)","(pollution, 0.9587423801422119)","(hydrosoluble, 0.9586848020553589)"
4,"(avaient, 0.9476783275604248)","(département, 0.9733659029006958)","(ulcère, 0.9178933501243591)","(perfusion, 0.9554687142372131)","(rhône, 0.9572724103927612)"
5,"(recevaient, 0.9445028901100159)","(hautement, 0.9717925190925598)","(intolérance, 0.9151709675788879)","(évolution, 0.9476815462112427)","(crâne, 0.9568729400634766)"
6,"(appartient, 0.9438688158988953)","(étroitement, 0.9711019992828369)","(séance, 0.9105365872383118)","(diminution, 0.9325371980667114)","(hexane, 0.9568067789077759)"
7,"(obtiennent, 0.9401053786277771)","(fortement, 0.970045804977417)","(résonance, 0.9102089405059814)","(microdélétion, 0.9222883582115173)","(lane, 0.9533616304397583)"
8,"(présentaient, 0.9388955235481262)","(épuisement, 0.968585193157196)","(paire, 0.9088973999023438)","(redistribution, 0.9204285144805908)","(flurbiprophène, 0.9522646069526672)"
9,"(soient, 0.9357691407203674)","(écoulement, 0.9675564765930176)","(caractère, 0.9088066816329956)","(distribution, 0.9199151992797852)","(eledone, 0.9520083665847778)"


In [187]:
press_model_results = get_most_similar_words(press_models, cadidat_words)
cbow_similar_words_press = pd.DataFrame(press_model_results[0])
cbow_similar_words_press

,patient,traitement,maladie,solution,jaune
0,"(concessionnaire, 0.606904923915863)","(coût, 0.6221883893013)","(population, 0.5803501009941101)","(alternative, 0.6218371987342834)","(maillot, 0.7428181767463684)"
1,"(sachet, 0.5687390565872192)","(financement, 0.6048182845115662)","(garantie, 0.573263943195343)","(suffisante, 0.6144033074378967)","(baden, 0.691763162612915)"
2,"(rescapé, 0.5582017302513123)","(collectif, 0.6001995801925659)","(mondialisation, 0.5651432871818542)","(démarche, 0.5852969288825989)","(cooke, 0.6716529726982117)"
3,"(cancéreux, 0.5455416440963745)","(système, 0.5983784794807434)","(douleur, 0.5566950440406799)","(règle, 0.5783204436302185)","(mcgee, 0.6622493267059326)"
4,"(peluche, 0.5442606806755066)","(logement, 0.5735400319099426)","(découverte, 0.556238055229187)","(foire, 0.567948579788208)","(saâdoune, 0.6396307945251465)"
5,"(crapuleux, 0.53504478931427)","(outil, 0.5537616014480591)","(vente, 0.5482550859451294)","(refonte, 0.5660814642906189)","(lachhab, 0.626640260219574)"
6,"(payait, 0.5346957445144653)","(vaccin, 0.5479769706726074)","(pension, 0.5434515476226807)","(dimension, 0.5603728294372559)","(etincelles, 0.6256155371665955)"
7,"(ordinateur, 0.5272122621536255)","(sida, 0.546791672706604)","(dot, 0.5398918986320496)","(bataille, 0.5576580166816711)","(bradeley, 0.6242194771766663)"
8,"(trou, 0.5232610702514648)","(cancer, 0.543422281742096)","(guérilla, 0.5359864830970764)","(firme, 0.5558780431747437)","(tho, 0.6209030151367188)"
9,"(clausus, 0.5150514841079712)","(mariage, 0.5266887545585632)","(séquence, 0.5342268347740173)","(notion, 0.551918625831604)","(croate, 0.613440752029419)"


In [188]:
skipgram_similar_words_press = pd.DataFrame(press_model_results[1])
skipgram_similar_words_press                                    

,patient,traitement,maladie,solution,jaune
0,"(cancéreux, 0.6621387004852295)","(médicamenteux, 0.5765436291694641)","(alzheimer, 0.6825127005577087)","(lancinant, 0.6425368189811707)","(maillot, 0.7617222666740417)"
1,"(humble, 0.6400861740112305)","(mineur, 0.57044917345047)","(pneumopathie, 0.6552125215530396)","(consensuelle, 0.6402616500854492)","(emparé, 0.6850003600120544)"
2,"(hospitalisé, 0.6261649131774902)","(allégations, 0.5510663390159607)","(orpheline, 0.635674238204956)","(garantissant, 0.6131341457366943)","(baden, 0.6657080054283142)"
3,"(irrecevabilité, 0.6195172071456909)","(disponibilité, 0.5401251912117004)","(pulmonaire, 0.6323383450508118)","(acceptable, 0.6079440116882324)","(pois, 0.6652892827987671)"
4,"(consterné, 0.6153663992881775)","(générateurs, 0.5377691388130188)","(virale, 0.6230897307395935)","(skier, 0.5891870856285095)","(grimpeur, 0.6600932478904724)"
5,"(maternel, 0.6097249388694763)","(anti-douleur, 0.5347406268119812)","(cancéreuse, 0.6218454241752625)","(agréée, 0.5803945064544678)","(pena, 0.6547910571098328)"
6,"(kilowattheure, 0.6065080761909485)","(antidouleur, 0.5313007235527039)","(famine, 0.6206404566764832)","(suffisante, 0.5790681838989258)","(grenadine, 0.653680145740509)"
7,"(soignant, 0.6056551337242126)","(redevenu, 0.5294235348701477)","(neurologique, 0.6180287599563599)","(cochonneries, 0.5788629651069641)","(brousse, 0.6396275162696838)"
8,"(relevée, 0.6056004762649536)","(enracinement, 0.5219552516937256)","(atypique, 0.6094968914985657)","(pacifique, 0.569839596748352)","(décaleront, 0.6352607607841492)"
9,"(éduqué, 0.6049503087997437)","(viol, 0.5219161510467529)","(assurance, 0.6075159907341003)","(sodium, 0.569588303565979)","(cooke, 0.63157057762146)"


In [189]:
fasttext_similar_words_press = pd.DataFrame(press_model_results[2])
fasttext_similar_words_press

,patient,traitement,maladie,solution,jaune
0,"(patientent, 0.9661645889282227)","(retraitement, 0.9317524433135986)","(malade, 0.8241892457008362)","(résolution, 0.9628271460533142)","(saâdoune, 0.8328008055686951)"
1,"(impatient, 0.9554916024208069)","(subitement, 0.9212613105773926)","(malnutrie, 0.7828246355056763)","(dissolution, 0.9603294134140015)","(lagune, 0.8215622901916504)"
2,"(détient, 0.9367938041687012)","(recrutement, 0.9102584719657898)","(magie, 0.7404280304908752)","(dilution, 0.9345404505729675)","(saadoune, 0.8149963021278381)"
3,"(impatientent, 0.9324398040771484)","(gratuitement, 0.9083817601203918)","(mala, 0.7223172783851624)","(pollution, 0.9282642602920532)","(dune, 0.8139479160308838)"
4,"(pataugent, 0.9268912672996521)","(bêtement, 0.9049776792526245)","(cancérologie, 0.719758927822113)","(évolution, 0.9255285263061523)","(brune, 0.8124436736106873)"
5,"(initient, 0.9248976707458496)","(dépècement, 0.9048676490783691)","(malawi, 0.718098521232605)","(révolution, 0.9244880676269531)","(lune, 0.8076838850975037)"
6,"(obtient, 0.9233004450798035)","(vêtement, 0.9042429327964783)","(trilogie, 0.7176809310913086)","(caution, 0.9049426913261414)","(abdoune, 0.805928111076355)"
7,"(renient, 0.9212331771850586)","(doctement, 0.9009862542152405)","(maîtrise, 0.717009961605072)","(résorption, 0.893635630607605)","(aune, 0.7804346084594727)"
8,"(abstient, 0.9184690713882446)","(logement, 0.8993222117424011)","(mélodie, 0.7135241031646729)","(persécution, 0.8838314414024353)","(l'une, 0.7704646587371826)"
9,"(remanient, 0.915896475315094)","(fondement, 0.8974041938781738)","(maladies, 0.7130897045135498)","(diminution, 0.8760914206504822)","(laâyoune, 0.764689028263092)"
